In [1]:
import os

from pathlib import Path

import matplotlib.pyplot as plt
import torch
import hashlib

from torch_geometric.data import Data, Dataset as PyGDataset

from qqe.GNN.physics_aware_NN import GNN, QuantumCircuitGraphDataset
import torch.nn as nn
from torch_geometric.loader import DataLoader

In [3]:
model_global = torch.load("../models/gnn_model_global.pt", map_location=torch.device('cpu'), weights_only=True)
model_cliff = torch.load("../models/gnn_model_clifford.pt", map_location=torch.device('cpu'), weights_only=True)
model_haar = torch.load("../models/gnn_model_haar.pt", map_location=torch.device('cpu'), weights_only=True)
model_quan = torch.load("../models/gnn_model_quansistor.pt", map_location=torch.device('cpu'), weights_only=True)
model_rand = torch.load("../models/gnn_model_random.pt", map_location=torch.device('cpu'), weights_only=True)

In [4]:
model_global

{'model_state_dict': OrderedDict([('conv_layers.0.lin_key.weight',
               tensor([[-0.1628, -0.0611, -0.0289,  ..., -0.0235,  0.1664,  0.0172],
                       [-0.1992,  0.1979,  0.7882,  ...,  0.0052,  0.0203, -0.0683],
                       [ 0.1773,  0.1598,  0.3580,  ...,  0.0402, -0.0179, -0.0989],
                       ...,
                       [ 0.0248,  0.1353, -0.2128,  ...,  0.0205,  0.0323, -0.0791],
                       [-0.0239, -0.0858, -0.0399,  ..., -0.2901, -0.0766,  0.0862],
                       [ 0.0392, -0.1032, -0.5756,  ..., -0.2340, -0.0973, -0.3937]])),
              ('conv_layers.0.lin_key.bias',
               tensor([-9.4148e-02, -4.9775e-03,  4.8915e-02,  1.3685e-01, -1.0517e-01,
                       -2.6321e-03, -9.6765e-02, -2.0303e-01, -1.5378e-01, -6.3220e-02,
                        1.3832e-01,  7.8766e-02, -1.4162e-01, -2.0665e-01, -9.2400e-02,
                        1.9488e-01, -1.6913e-01,  9.3842e-03,  1.6121e-01,  9.3481e

In [5]:
model_cliff

{'model_state_dict': OrderedDict([('conv_layers.0.lin_key.weight',
               tensor([[ 0.0172, -0.1633,  0.0092,  ...,  0.1165, -0.0710, -0.1990],
                       [-0.1047,  0.0422,  0.3499,  ...,  0.1588, -0.1267,  0.0111],
                       [-0.1595,  0.0175,  0.3035,  ..., -0.1548, -0.2756, -0.0727],
                       ...,
                       [ 0.0920,  0.1971,  0.0036,  ..., -0.1476, -0.1123,  0.2426],
                       [ 0.0875,  0.2268, -0.1706,  ..., -0.1004, -0.1500, -0.0891],
                       [ 0.2065, -0.0319,  0.1331,  ...,  0.0269, -0.1515, -0.1303]])),
              ('conv_layers.0.lin_key.bias',
               tensor([ 0.2139,  0.1764, -0.0217,  0.1926, -0.0402,  0.1581, -0.1077, -0.1067,
                       -0.1644, -0.2116,  0.0946, -0.0823,  0.0272, -0.1482, -0.1449, -0.0005,
                        0.1807,  0.0983,  0.2336,  0.1154, -0.0183,  0.0504, -0.0915,  0.0549,
                        0.1466,  0.1443,  0.1790,  0.0363,  0.

In [6]:
model_quan

{'model_state_dict': OrderedDict([('conv_layers.0.lin_key.weight',
               tensor([[ 0.1395,  0.1772, -0.1824,  ...,  0.2155, -0.2172,  0.0769],
                       [ 0.2436, -0.1896, -0.1619,  ...,  0.2286,  0.0968,  0.0078],
                       [-0.0914,  0.1742, -0.0013,  ..., -0.1833,  0.0734, -0.1093],
                       ...,
                       [ 0.2223,  0.1559, -0.1463,  ...,  0.1651,  0.3006, -0.1690],
                       [-0.1848,  0.1279,  0.1979,  ..., -0.2600,  0.0298, -0.1634],
                       [ 0.0295,  0.0888,  0.1203,  ...,  0.1210,  0.0044,  0.0576]])),
              ('conv_layers.0.lin_key.bias',
               tensor([ 0.0307, -0.0697,  0.2164,  0.2140, -0.1621,  0.0121,  0.2529, -0.2319,
                       -0.0825,  0.2340,  0.2527, -0.1671,  0.2433,  0.1468,  0.1531,  0.0422,
                        0.0452,  0.0956, -0.0167,  0.0433, -0.2631,  0.0647, -0.1172,  0.1719,
                        0.2054,  0.2491,  0.1874, -0.2547,  0.

In [7]:
model_rand

{'model_state_dict': OrderedDict([('conv_layers.0.lin_key.weight',
               tensor([[-0.1978, -0.0564, -0.0804,  ..., -0.0029, -0.1609,  0.2378],
                       [ 0.1339,  0.2426, -0.0541,  ..., -0.2640, -0.0258, -0.1433],
                       [-0.1883, -0.2133, -0.0565,  ...,  0.1195,  0.0289, -0.2947],
                       ...,
                       [-0.2259,  0.0115,  0.0859,  ...,  0.1660, -0.0509, -0.1800],
                       [-0.1920, -0.1664, -0.2198,  ...,  0.1848, -0.1360,  0.0289],
                       [-0.1177, -0.1554,  0.2219,  ...,  0.0713, -0.1562, -0.2067]])),
              ('conv_layers.0.lin_key.bias',
               tensor([-0.0325,  0.0843, -0.0465, -0.2014, -0.0674, -0.2271, -0.2260, -0.1182,
                        0.2279, -0.2095,  0.1508, -0.1025, -0.1887,  0.0234, -0.2217,  0.0615,
                       -0.0772,  0.1851,  0.1601,  0.1927,  0.1989, -0.2029,  0.2009,  0.0055,
                       -0.0038, -0.1249,  0.2057, -0.2311,  0.

In [8]:
model_haar

{'model_state_dict': OrderedDict([('conv_layers.0.lin_key.weight',
               tensor([[ 0.1485,  0.2177, -0.2088,  ...,  0.0236, -0.1249,  0.1551],
                       [ 0.0282,  0.0492,  0.0780,  ...,  0.2038,  0.2074,  0.0593],
                       [-0.1807,  0.1725,  0.0234,  ..., -0.1743, -0.0577, -0.2018],
                       ...,
                       [-0.2039, -0.1072,  0.1900,  ...,  0.2977, -0.1275,  0.1074],
                       [ 0.2686,  0.0327, -0.2288,  ..., -0.0062, -0.0318,  0.0294],
                       [ 0.1743,  0.2115, -0.1778,  ..., -0.0847, -0.2043,  0.0245]])),
              ('conv_layers.0.lin_key.bias',
               tensor([-1.5489e-01,  1.7029e-02,  6.5179e-02,  3.9576e-02,  4.8571e-02,
                       -3.6019e-02,  2.1864e-01, -6.7556e-02, -1.5803e-01,  1.9529e-01,
                       -5.7184e-02,  1.1910e-01, -1.0923e-01, -1.0489e-01,  2.6485e-01,
                       -2.3196e-01,  2.7605e-01,  1.4630e-01,  1.3546e-01,  2.0475e

In [8]:
def collect_pred_paths(dataset_dir: str, family: str | None = None) -> list[str]:
    d = Path(dataset_dir)
    pred_root = d / "predictions"

    if family is not None:
        paths = sorted((pred_root / family).glob("*.pt"))
    else:
        paths = []
        if pred_root.exists():
            for family_dir in sorted(pred_root.iterdir()):
                if family_dir.is_dir():
                    paths.extend(sorted(family_dir.glob("*.pt")))

    if not paths:
        paths = sorted(d.glob("*.pt"))

    return [str(p.resolve()) for p in paths]


def _cache_root_for_paths(paths: list[str], suffix: str = "") -> str:
    # Include full resolved paths in the digest to avoid collisions across folders/families.
    canonical = "|".join(sorted(str(Path(p).resolve()) for p in paths))
    digest = hashlib.md5(canonical.encode("utf-8")).hexdigest()[:10]
    tag = f"_{suffix}" if suffix else ""
    cache_dir = Path("..") / "qqe" / "cache" / f"pyg_cache_{digest}{tag}"
    cache_dir.mkdir(parents=True, exist_ok=True)
    return str(cache_dir.resolve())

In [9]:
class PaddedGraphDatasetWrapper:
    """Wrapper that pads/truncates graph features to consistent dimensions."""

    def __init__(
        self,
        dataset,
        target_node_dim: int | None = None,
        target_global_dim: int | None = None,
        target_dim: int | None = None,  # backwards compatibility
    ):
        self.dataset = dataset

        # Backwards compatibility with older call sites using target_dim.
        if target_node_dim is None and target_dim is not None:
            target_node_dim = target_dim

        self.target_dim = target_node_dim if target_node_dim is not None else self._compute_max_node_dim()
        self.target_global_dim = (
            target_global_dim if target_global_dim is not None else self._compute_max_global_dim()
        )

    def _compute_max_node_dim(self) -> int:
        """Find max node feature width across all samples."""
        max_dim = 0
        for i in range(len(self.dataset)):
            data = self.dataset[i]
            x = getattr(data, "x", None)
            if x is not None and x.dim() > 1:
                max_dim = max(max_dim, int(x.shape[1]))
        return max_dim

    def _compute_max_global_dim(self) -> int:
        """Find max global feature width across all samples."""
        max_dim = 0
        for i in range(len(self.dataset)):
            data = self.dataset[i]
            g = getattr(data, "global_features", None)
            if g is None:
                continue
            if g.dim() == 0:
                width = 1
            elif g.dim() == 1:
                width = int(g.shape[0])
            else:
                width = int(g.shape[-1])
            max_dim = max(max_dim, width)
        return max_dim

    def _fit_node_dim(self, data):
        x = getattr(data, "x", None)
        if x is None or x.dim() <= 1:
            return data
        current = int(x.shape[1])
        if current == self.target_dim:
            return data
        out = data.clone()
        if current < self.target_dim:
            pad_size = self.target_dim - current
            out.x = torch.nn.functional.pad(out.x, (0, pad_size), mode="constant", value=0.0)
        else:
            out.x = out.x[:, : self.target_dim]
        return out

    def _fit_global_dim(self, data):
        g = getattr(data, "global_features", None)
        if g is None or self.target_global_dim <= 0:
            return data

        out = data.clone() if out_is_same(data, g) else data
        g = getattr(out, "global_features")

        # Ensure graph-level vector shape.
        if g.dim() == 0:
            g = g.view(1)
        elif g.dim() > 1:
            g = g.view(-1)

        current = int(g.shape[0])
        if current < self.target_global_dim:
            g = torch.nn.functional.pad(
                g, (0, self.target_global_dim - current), mode="constant", value=0.0
            )
        elif current > self.target_global_dim:
            g = g[: self.target_global_dim]

        out.global_features = g
        return out

    def __getitem__(self, idx: int):
        data = self.dataset[idx]
        data = self._fit_node_dim(data)
        data = self._fit_global_dim(data)
        return data

    def __len__(self) -> int:
        return len(self.dataset)


def out_is_same(data, g):
    # Clone lazily only when we actually need to edit global features.
    return hasattr(data, "global_features") and data.global_features is g

In [10]:
def build_pred_loaders_two_stage(
    pt_paths: list[str],
    batch_size: int = 32,
    seed: int = 42,
    global_feature_variant: str = "baseline",
    node_feature_backend_variant: str | None = None,
) -> tuple[QuantumCircuitGraphDataset, PaddedGraphDatasetWrapper,DataLoader, int, int]:
    suffix = f"{global_feature_variant}_backend_{node_feature_backend_variant or 'none'}"
    root = _cache_root_for_paths(pt_paths, suffix=suffix)

    dataset = QuantumCircuitGraphDataset(
        root=root,
        pt_paths=pt_paths,
        global_feature_variant=global_feature_variant,
        node_feature_backend_variant=node_feature_backend_variant,
    )

    if len(dataset) < 3:
        raise RuntimeError("Dataset too small for train/val/test splitting.")

    padded_dataset = PaddedGraphDatasetWrapper(dataset)
    node_in_dim = padded_dataset.target_dim
    global_in_dim = dataset.global_feature_dim

    pred_ds = padded_dataset
    pin_mem = torch.cuda.is_available()
    return (
        dataset,
        padded_dataset,
        DataLoader(pred_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=pin_mem),
        node_in_dim,
        global_in_dim,
    )

In [67]:
family = "haar"
global_feature_variant = "binned"
node_feature_backend_variant = None

In [82]:
MODEL_STATE_PATH = f"../models/gnn_model_{family}.pt"
CHECKPOINT_PATH = f"../models/gnn_model_{family}.pt"

In [83]:
pred_data_paths = collect_pred_paths("../outputs/data", family=family)
print(f"Collected {len(pred_data_paths)} .pt files for dataset.")

Collected 8750 .pt files for dataset.


In [84]:
dataset, padded_data, pred_loader, node_in_dim, global_in_dim = build_pred_loaders_two_stage(
    pred_data_paths,  # Use first 10 paths for demonstration
    batch_size=32,
)

In [92]:
dataset[0]

Data(
  x=[85, 25],
  edge_index=[2, 134],
  y=[1],
  global_features=[1, 3],
  num_qubits=12,
  gate_counts={ haar_count=61 },
  meta={
    cid='haar_Q12_L11_S1220274776',
    family='haar',
    n_qubits=12,
    n_layers=11,
    seed=1220274776,
    backend='pennylane',
    method='fwht',
    representation='dense',
    n_bins=50,
  }
)

In [85]:
# --------------------------------------------------
# 1. Build prediction dataset with training-compatible feature config
# --------------------------------------------------
if not pred_data_paths:
    raise RuntimeError("No prediction .pt files found. Check outputs/data/predictions/<family>.")

checkpoint = None
model_config = {}
fixed_all_gate_keys = None

if Path(CHECKPOINT_PATH).exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
    if isinstance(checkpoint, dict):
        model_config = checkpoint.get("model_config", {}) or {}
        feature_config = checkpoint.get("feature_config", {}) or {}
        fixed_all_gate_keys = feature_config.get("all_gate_keys", None)

suffix = f"{global_feature_variant}_backend_{node_feature_backend_variant or 'none'}"
root = _cache_root_for_paths(pred_data_paths, suffix=suffix)

pred_dataset = QuantumCircuitGraphDataset(
    root=root,
    pt_paths=pred_data_paths,
    global_feature_variant=global_feature_variant,
    node_feature_backend_variant=node_feature_backend_variant,
    fixed_all_gate_keys=fixed_all_gate_keys,
)

trained_node_in_dim = model_config.get("node_in_dim", None)
trained_global_in_dim = model_config.get("global_in_dim", None)

padded_pred_dataset = PaddedGraphDatasetWrapper(
    pred_dataset,
    target_node_dim=trained_node_in_dim if trained_node_in_dim is not None else None,
    target_global_dim=trained_global_in_dim if trained_global_in_dim is not None else None,
    target_dim=trained_node_in_dim if trained_node_in_dim is not None else None,
 )

pred_loader = DataLoader(
    padded_pred_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

node_in_dim = padded_pred_dataset.target_dim
global_in_dim = padded_pred_dataset.target_global_dim

print(f"Prediction graphs: {len(pred_dataset)}")
print(f"Prediction node feature dim: {node_in_dim}")
print(f"Prediction global feature dim: {global_in_dim}")
if trained_node_in_dim is not None or trained_global_in_dim is not None:
    print(f"Trained node/global dims from checkpoint: {trained_node_in_dim}/{trained_global_in_dim}")

Prediction graphs: 8750
Prediction node feature dim: 23
Prediction global feature dim: 3
Trained node/global dims from checkpoint: 23/3


C:\Users\Victor\AppData\Local\Temp\ipykernel_187708\351848649.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu

In [86]:
pred_dataset[0]

Data(
  x=[85, 25],
  edge_index=[2, 134],
  y=[1],
  global_features=[1, 3],
  num_qubits=12,
  gate_counts={ haar_count=61 },
  meta={
    cid='haar_Q12_L11_S1220274776',
    family='haar',
    n_qubits=12,
    n_layers=11,
    seed=1220274776,
    backend='pennylane',
    method='fwht',
    representation='dense',
    n_bins=50,
  }
)

In [87]:
# --------------------------------------------------
# 2. Validate dataset/model dimensions
# --------------------------------------------------
print("node_in_dim (prediction) =", node_in_dim)
print("global_in_dim (prediction) =", global_in_dim)
print("node_in_dim (trained) =", trained_node_in_dim)
print("global_in_dim (trained) =", trained_global_in_dim)

if trained_node_in_dim is not None and node_in_dim != trained_node_in_dim:
    print(f"Node dim will be adapted to trained dim {trained_node_in_dim} at inference time.")

if trained_global_in_dim is not None and global_in_dim != trained_global_in_dim:
    print(f"Global feature dim will be adapted to trained dim {trained_global_in_dim} at inference time.")

node_in_dim (prediction) = 23
global_in_dim (prediction) = 3
node_in_dim (trained) = 23
global_in_dim (trained) = 3


In [88]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [89]:
# --------------------------------------------------
# 3. Rebuild model and load weights robustly
# --------------------------------------------------
def _extract_state_dict(payload):
    if isinstance(payload, nn.Module):
        return payload.state_dict()
    if isinstance(payload, dict) and "model_state_dict" in payload:
        return payload["model_state_dict"]
    if isinstance(payload, dict) and all(torch.is_tensor(v) for v in payload.values()):
        return payload
    raise RuntimeError("Unsupported model file format.")

if checkpoint is not None and isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    if not Path(MODEL_STATE_PATH).exists():
        raise FileNotFoundError(f"Could not find model weights at {MODEL_STATE_PATH}")
    raw_payload = torch.load(MODEL_STATE_PATH, map_location="cpu")
    state_dict = _extract_state_dict(raw_payload)

model_kwargs = {
    "node_in_dim": int(trained_node_in_dim if trained_node_in_dim is not None else node_in_dim),
    "global_in_dim": int(trained_global_in_dim if trained_global_in_dim is not None else global_in_dim),
    "gnn_hidden": int(model_config.get("gnn_hidden", 32)),
    "gnn_heads": int(model_config.get("gnn_heads", 8)),
    "global_hidden": int(model_config.get("global_hidden", 16)),
    "reg_hidden": int(model_config.get("reg_hidden", 16)),
    "num_layers": int(model_config.get("num_layers", 5)),
    "dropout_rate": float(model_config.get("dropout_rate", 0.1)),
}

model = GNN(**model_kwargs).to(device)
missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
model.eval()

print("Loaded model config:", model_kwargs)
if missing_keys:
    print("Missing keys:", missing_keys)
if unexpected_keys:
    print("Unexpected keys:", unexpected_keys)

Loaded model config: {'node_in_dim': 23, 'global_in_dim': 3, 'gnn_hidden': 32, 'gnn_heads': 8, 'global_hidden': 16, 'reg_hidden': 16, 'num_layers': 5, 'dropout_rate': 0.1}


In [90]:
# --------------------------------------------------
# 4. Predict with defensive dimension adaptation
# --------------------------------------------------
predictions = []
expected_node_dim = model_kwargs["node_in_dim"]
expected_global_dim = model_kwargs["global_in_dim"]

with torch.no_grad():
    for batch in pred_loader:
        # Adapt node features to the model input dimension.
        if batch.x.size(1) < expected_node_dim:
            pad_size = expected_node_dim - batch.x.size(1)
            batch.x = torch.nn.functional.pad(batch.x, (0, pad_size), mode="constant", value=0.0)
        elif batch.x.size(1) > expected_node_dim:
            batch.x = batch.x[:, :expected_node_dim]

        # Adapt global features to the model input dimension.
        g = batch.global_features
        if g.dim() == 1:
            g = g.view(batch.num_graphs, -1)
        if g.size(1) < expected_global_dim:
            g = torch.nn.functional.pad(g, (0, expected_global_dim - g.size(1)), mode="constant", value=0.0)
        elif g.size(1) > expected_global_dim:
            g = g[:, :expected_global_dim]
        batch.global_features = g

        batch = batch.to(device)
        pred = model(batch).view(-1)
        predictions.extend(pred.cpu().tolist())

print("First 10 predictions:", predictions[:10])
print("Total predictions:", len(predictions))

First 10 predictions: [7.202025413513184, 7.202025890350342, 7.202024936676025, 7.202025413513184, 7.202025413513184, 7.202025413513184, 7.202025413513184, 7.202024936676025, 7.202025413513184, 7.202025890350342]
Total predictions: 8750


In [77]:
# --------------------------------------------------
# 4. Predict with defensive dimension adaptation
# --------------------------------------------------
predictions = []
expected_node_dim = model_kwargs["node_in_dim"]
expected_global_dim = model_kwargs["global_in_dim"]

with torch.no_grad():
    for batch in pred_loader:
        # Adapt node features to the model input dimension.
        if batch.x.size(1) < expected_node_dim:
            pad_size = expected_node_dim - batch.x.size(1)
            batch.x = torch.nn.functional.pad(batch.x, (0, pad_size), mode="constant", value=0.0)
        elif batch.x.size(1) > expected_node_dim:
            batch.x = batch.x[:, :expected_node_dim]

        # Adapt global features to the model input dimension.
        g = batch.global_features
        if g.dim() == 1:
            g = g.view(batch.num_graphs, -1)
        if g.size(1) < expected_global_dim:
            g = torch.nn.functional.pad(g, (0, expected_global_dim - g.size(1)), mode="constant", value=0.0)
        elif g.size(1) > expected_global_dim:
            g = g[:, :expected_global_dim]
        batch.global_features = g

        batch = batch.to(device)
        pred = model(batch).view(-1)
        predictions.extend(pred.cpu().tolist())

print("First 10 predictions:", predictions[:10])
print("Total predictions:", len(predictions))

First 10 predictions: [7.105358600616455, 7.105358600616455, 7.105358600616455, 7.105359077453613, 7.105358600616455, 7.105358600616455, 7.105358123779297, 7.105358600616455, 7.105358600616455, 7.105358600616455]
Total predictions: 8750


In [91]:
predictions

[7.202025413513184,
 7.202025890350342,
 7.202024936676025,
 7.202025413513184,
 7.202025413513184,
 7.202025413513184,
 7.202025413513184,
 7.202024936676025,
 7.202025413513184,
 7.202025890350342,
 7.202024936676025,
 7.202025890350342,
 7.202025890350342,
 7.202025413513184,
 7.202025413513184,
 7.202025413513184,
 7.202024936676025,
 7.202025890350342,
 7.202025413513184,
 7.202025413513184,
 7.202025890350342,
 7.202025413513184,
 7.202025413513184,
 7.202025890350342,
 7.202025413513184,
 7.340615749359131,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.340615749359131,
 7.340615272521973,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.340615749359131,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,


In [78]:
def collect_pt_paths(dataset_dir: str, family: str | None = None) -> list[str]:
    d = Path(dataset_dir)
    # support either dataset_dir/*.pt or dataset_dir/samples/*.pt
    if family is not None:
        paths = sorted((d / "encoding_data_pennylane" / family).glob("*.pt"))
    else:
        paths = []
        encoding_dir = d / "encoding_data_pennylane"
        if encoding_dir.exists():
            for family_dir in sorted(encoding_dir.iterdir()):
                if family_dir.is_dir():
                    paths.extend(sorted(family_dir.glob("*.pt")))
    if not paths:
        paths = sorted(d.glob("*.pt"))
    return [str(p) for p in paths]

def _cache_root_for_paths(paths: list[str], suffix: str = "") -> str:
    canonical = "|".join(sorted(Path(p).name for p in paths))

    digest = hashlib.md5(canonical.encode("utf-8")).hexdigest()[:10]

    tag = f"_{suffix}" if suffix else ""

    cache_dir = Path("qqe") / "cache" / f"pyg_cache_{digest}{tag}"
    cache_dir.mkdir(parents=True, exist_ok=True)

    return str(cache_dir)

In [79]:
class PaddedGraphDatasetWrapper:
    """Wrapper that pads/truncates graph features to consistent dimensions."""

    def __init__(
        self,
        dataset,
        target_node_dim: int | None = None,
        target_global_dim: int | None = None,
        target_dim: int | None = None,  # backwards compatibility
    ):
        self.dataset = dataset

        # Backwards compatibility with older call sites using target_dim.
        if target_node_dim is None and target_dim is not None:
            target_node_dim = target_dim

        self.target_dim = target_node_dim if target_node_dim is not None else self._compute_max_node_dim()
        self.target_global_dim = (
            target_global_dim if target_global_dim is not None else self._compute_max_global_dim()
        )

    def _compute_max_node_dim(self) -> int:
        """Find max node feature width across all samples."""
        max_dim = 0
        for i in range(len(self.dataset)):
            data = self.dataset[i]
            x = getattr(data, "x", None)
            if x is not None and x.dim() > 1:
                max_dim = max(max_dim, int(x.shape[1]))
        return max_dim

    def _compute_max_global_dim(self) -> int:
        """Find max global feature width across all samples."""
        max_dim = 0
        for i in range(len(self.dataset)):
            data = self.dataset[i]
            g = getattr(data, "global_features", None)
            if g is None:
                continue
            if g.dim() == 0:
                width = 1
            elif g.dim() == 1:
                width = int(g.shape[0])
            else:
                width = int(g.shape[-1])
            max_dim = max(max_dim, width)
        return max_dim

    def _fit_node_dim(self, data):
        x = getattr(data, "x", None)
        if x is None or x.dim() <= 1:
            return data
        current = int(x.shape[1])
        if current == self.target_dim:
            return data
        out = data.clone()
        if current < self.target_dim:
            pad_size = self.target_dim - current
            out.x = torch.nn.functional.pad(out.x, (0, pad_size), mode="constant", value=0.0)
        else:
            out.x = out.x[:, : self.target_dim]
        return out

    def _fit_global_dim(self, data):
        g = getattr(data, "global_features", None)
        if g is None or self.target_global_dim <= 0:
            return data

        out = data.clone() if out_is_same(data, g) else data
        g = getattr(out, "global_features")

        # Ensure graph-level vector shape.
        if g.dim() == 0:
            g = g.view(1)
        elif g.dim() > 1:
            g = g.view(-1)

        current = int(g.shape[0])
        if current < self.target_global_dim:
            g = torch.nn.functional.pad(
                g, (0, self.target_global_dim - current), mode="constant", value=0.0
            )
        elif current > self.target_global_dim:
            g = g[: self.target_global_dim]

        out.global_features = g
        return out

    def __getitem__(self, idx: int):
        data = self.dataset[idx]
        data = self._fit_node_dim(data)
        data = self._fit_global_dim(data)
        return data

    def __len__(self) -> int:
        return len(self.dataset)


def out_is_same(data, g):
    # Clone lazily only when we actually need to edit global features.
    return hasattr(data, "global_features") and data.global_features is g

In [80]:
data_paths = collect_pt_paths("../outputs/data", family=None)
print(f"Collected {len(data_paths)} .pt files for dataset.")

Collected 40000 .pt files for dataset.
